# Generate Diagnostics for Shuffled Numerical States

This Generate notebook computes and caches the data consumed by `Visualize-shuffled-layer-averaged-diagnostics-across-context-lengths.ipynb`. The companion Visualize notebook shows layer-averaged graph-spectral diagnostics across context lengths for shuffled numerical states using Llama-3.2-3B, as presented under **Additional Controls and Higher-Dimensional Inputs** in the appendix.

This control tests whether token diversity alone explains the observed organization. For each realization and context length, numerical states are randomly permuted within the selected prefix, preserving exact token identities, frequencies, and separator positions. Constant inputs are omitted because shuffling leaves them unchanged.


Set imports, constants, output paths, sweep axes, and random seeds.

In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import gc
import hashlib
import json
import random
from pathlib import Path
import numpy as np
import torch
from scipy.integrate import solve_ivp
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model and sweep configuration
SEED = 42
ENSEMBLE_SEED = SEED
# Use base seed 42 with a separate per-task generator from trajectory generation
SHUFFLE_SEED = 42
SHUFFLE_METHOD = "numeric_token_permutation_within_prefix_fixed_bos_and_commas_v1"
MODEL_NAME = "meta-llama/Llama-3.2-3B"
EXPECTED_NUM_LAYERS = 28
NUM_ENSEMBLE_RUNS = 20
EXPECTED_NUM_INPUT_FAMILIES = 9

# Diagnostics computed at each context length
MAX_CONTEXT_STATES = 1000
TOTAL_SEQUENCE_STATES = MAX_CONTEXT_STATES
# Checkpoint progress; save final plot data to OUTPUT_PATH
RESUME_FROM_CHECKPOINT = True
CHECKPOINT_EVERY_N_TASKS = 20

# Use paths relative to this notebook folder
GENERATION_NOTEBOOK_NAME = "Generate-shuffled-layer-averaged-diagnostics-across-context-lengths.ipynb"
NOTEBOOK_DIR = Path(".")
if not (NOTEBOOK_DIR / GENERATION_NOTEBOOK_NAME).is_file():
    raise FileNotFoundError("Run this notebook from its containing folder.")
CACHE_DIR = NOTEBOOK_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_PATH = CACHE_DIR / "shuffled_diagnostics_averaged_over_layers_vs_context_length_20_runs.npz"
CHECKPOINT_PATH = CACHE_DIR / "shuffled_diagnostics_averaged_over_layers_vs_context_length_20_runs.resume_checkpoint.npz"

NPZ_SCHEMA_VERSION = "shuffled_three_diagnostics_layer_averaged_v1"
CHECKPOINT_SCHEMA_VERSION = NPZ_SCHEMA_VERSION + "_checkpoint"

DROP_BOS = True
RENORMALIZE_AFTER_DROP = True
# Mass and uniform weighting are equivalent for the models studied here; mass also supports sparse-attention variants
HEAD_WEIGHTING = "mass"
TOKENIZATION_ADD_SPECIAL_TOKENS = True
K_CUTOFF = None

# Sweep axes
context_fracs_full = np.asarray([i / 20 for i in range(1, 21)], dtype=np.float64)
context_percent_full = np.asarray([int(round(100 * frac)) for frac in context_fracs_full], dtype=np.int64)
run_ids_full = np.arange(NUM_ENSEMBLE_RUNS, dtype=np.int64)

input_data_order_full = np.asarray([
    "Constant sampled",
    "Logistic r=3.2",
    "Logistic r=3.5",
    "Logistic r=3.55",
    "Logistic r=3.566",
    "Logistic r=3.5692",
    "Logistic r=3.5698",
    "Logistic r=3.9",
    "Logistic r=4.0",
    "Lorenz",
], dtype=str)

# Preserve the original full-family indices for reproducible shuffle seeds
# Constant inputs are identical after shuffling; skip their model evaluations
active_input_indices = np.flatnonzero(input_data_order_full != "Constant sampled")
active_run_indices = np.arange(run_ids_full.size, dtype=np.int64)
active_context_indices = np.arange(context_fracs_full.size, dtype=np.int64)

input_names = input_data_order_full[active_input_indices]
run_ids = run_ids_full[active_run_indices]
context_fracs = context_fracs_full[active_context_indices]
context_percent = context_percent_full[active_context_indices]

diagnostic_names = np.asarray([
    "layer_averaged_effective_active_modes",
    "layer_averaged_normalized_fiedler_value_lambda_2",
    "layer_averaged_hfer",
], dtype=str)

# Seed data generation and model inference
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


Define sequence generators and labels for the input families.

In [ ]:
# Periodic and chaotic logistic-map regimes
periodic_logistic_periods = {
    3.2: 2,
    3.5: 4,
    3.55: 8,
    3.566: 16,
    3.5692: 32,
    3.5698: 64,
}
logistic_expected_periods = {
    r: periodic_logistic_periods.get(r)
    for r in list(periodic_logistic_periods.keys()) + [3.9, 4.0]
}
logistic_long_n = 20000
logistic_settle_tol = 1e-10
periodic_random_phase = True

# Match Lorenz sampling to the sequence length
sigma = 10.0
rho = 28.0
beta = 8.0 / 3.0
lorenz_dt = 100 / 999
t_eval = np.arange(TOTAL_SEQUENCE_STATES, dtype=float) * lorenz_dt
t_span = (float(t_eval[0]), float(t_eval[-1]))


def lorenz(t, y, sigma, rho, beta):
    return [
        sigma * (y[1] - y[0]),
        y[0] * (rho - y[2]) - y[1],
        y[0] * y[1] - beta * y[2],
    ]


def logistic_sequence(r, x0, n):
    x = np.empty(int(n), dtype=float)
    x[0] = float(x0)
    for i in range(1, int(n)):
        x[i] = float(r) * x[i - 1] * (1 - x[i - 1])
    return x


def settle_time_float(x, p, tol=1e-10):
    x = np.asarray(x, dtype=float)
    if p is None or p <= 0 or p >= x.size:
        return None
    err = np.abs(x[p:] - x[:-p])
    suffix_max = np.maximum.accumulate(err[::-1])[::-1]
    hits = np.flatnonzero(suffix_max < tol)
    return int(hits[0]) if hits.size else None


def quantize_logistic_window(vals):
    vals = np.asarray(vals, dtype=float)
    vmin, vmax = vals.min(), vals.max()
    if vmax == vmin:
        return np.full(vals.shape[0], 500, dtype=int)
    return np.round(50 + (vals - vmin) / (vmax - vmin) * 900).astype(int)


def build_burned_in_logistic_window_ensemble(
    r,
    x0,
    n_window,
    expected_period=None,
    n_long=20000,
    tol=1e-10,
    phase_offset=0,
):
    # Use settled periodic windows with an optional phase shift
    if expected_period is None:
        vals = logistic_sequence(r, x0, n_window)
        return np.asarray(vals, dtype=float), 0, 0, False

    phase_offset = int(phase_offset) % int(expected_period)
    x_long = logistic_sequence(r, x0, n_long)
    t_settle = settle_time_float(x_long, expected_period, tol=tol)
    if t_settle is None:
        raise ValueError(f"Could not find settle time for r={r} with expected period {expected_period}.")

    start_idx = int(t_settle) + int(phase_offset)
    if start_idx + n_window > x_long.size:
        raise ValueError(
            f"Need a longer trajectory for r={r}; start index {start_idx} leaves fewer than {n_window} states."
        )

    vals = np.asarray(x_long[start_idx:start_idx + n_window], dtype=float)
    return vals, int(t_settle), int(phase_offset), True


def quantize_lorenz_x_values(x_values):
    x_values = np.asarray(x_values, dtype=float)
    x_min_rep = float(np.min(x_values))
    x_max_rep = float(np.max(x_values))
    if x_max_rep == x_min_rep:
        return np.full(x_values.shape[0], 500, dtype=int)
    return np.round(50 + (x_values - x_min_rep) / (x_max_rep - x_min_rep) * 900).astype(int)


def _display_name(data_name):
    # Cache labels for visualization
    display_names = {
        "Constant sampled": "Constant sampled",
        "Logistic r=3.2": "Logistic r=3.2 (period 2)",
        "Logistic r=3.5": "Logistic r=3.5 (period 4)",
        "Logistic r=3.55": "Logistic r=3.55 (period 8)",
        "Logistic r=3.566": "Logistic r=3.566 (period 16)",
        "Logistic r=3.5692": "Logistic r=3.5692 (period 32)",
        "Logistic r=3.5698": "Logistic r=3.5698 (period 64)",
        "Logistic r=3.9": "Logistic r=3.9 (chaotic)",
        "Logistic r=4.0": "Logistic r=4.0 (chaotic)",
        "Lorenz": "Lorenz",
    }
    return display_names.get(str(data_name), str(data_name))


def _input_class_label(data_name):
    data_name = str(data_name)
    if data_name == "Constant sampled":
        return "constant"
    if data_name in {f"Logistic r={r}" for r in periodic_logistic_periods}:
        return "periodic"
    if data_name in {"Logistic r=3.9", "Logistic r=4.0"}:
        return "chaotic"
    if data_name == "Lorenz":
        return "lorenz"
    raise ValueError(f"Unknown input family: {data_name}")


input_labels = np.asarray([_display_name(name) for name in input_names], dtype=str)
input_class_labels = np.asarray([_input_class_label(name) for name in input_names], dtype=str)

if input_names.shape != (EXPECTED_NUM_INPUT_FAMILIES,):
    raise ValueError(f"Expected {EXPECTED_NUM_INPUT_FAMILIES} inputs, got {input_names.shape}.")
if run_ids.shape != (NUM_ENSEMBLE_RUNS,):
    raise ValueError(f"Expected {NUM_ENSEMBLE_RUNS} run ids, got {run_ids.shape}.")
if context_fracs.shape != (20,):
    raise ValueError(f"Expected 20 context fractions, got {context_fracs.shape}.")



Build the ensemble of constant, logistic, and Lorenz token sequences.

In [ ]:
# Share each replicate's x0 across input families
ensemble_rng = np.random.default_rng(ENSEMBLE_SEED)
x0_samples = ensemble_rng.uniform(0.3, 0.7, size=NUM_ENSEMBLE_RUNS)
phase_rng = np.random.default_rng(ENSEMBLE_SEED + 1)

ensemble_periodic_rs = [3.2, 3.5, 3.55, 3.566, 3.5692, 3.5698]
ensemble_chaotic_rs = [3.9, 4.0]

constant_tokens = np.full(NUM_ENSEMBLE_RUNS, -1, dtype=np.int64)
ensemble_inputs = {}

for rep_id, x0_rep in enumerate(x0_samples):
    rep_id = int(rep_id)
    x0_rep = float(x0_rep)
    rep_inputs = {}

    lorenz_x0_rep = float(-8.0 + (x0_rep - 0.5))
    lorenz_y0_rep = 7.0
    lorenz_z0_rep = 27.0

    # Repeat one token per constant replicate
    constant_token = int(np.round(50 + 900 * x0_rep))
    constant_tokens[rep_id] = constant_token
    rep_inputs["Constant sampled"] = {
        "sequence": np.full(TOTAL_SEQUENCE_STATES, constant_token, dtype=int),
        "metadata": {
            "input_class": "constant",
            "rep_id": rep_id,
            "x0_rep": x0_rep,
            "constant_token": constant_token,
        },
    }

    # Use settled periodic windows with randomized phase
    for r in ensemble_periodic_rs:
        expected_period = logistic_expected_periods[r]
        if expected_period is None:
            raise ValueError(f"Expected a periodic window for r={r}, but expected_period is None.")
        phase_offset = (
            int(phase_rng.integers(0, expected_period))
            if periodic_random_phase
            else 0
        )
        vals, burn_in_start_index, used_phase_offset, used_burn_in = build_burned_in_logistic_window_ensemble(
            r,
            x0=x0_rep,
            n_window=TOTAL_SEQUENCE_STATES,
            expected_period=expected_period,
            n_long=logistic_long_n,
            tol=logistic_settle_tol,
            phase_offset=phase_offset,
        )
        data_name = f"Logistic r={r}"
        rep_inputs[data_name] = {
            "sequence": quantize_logistic_window(vals),
            "metadata": {
                "input_class": "periodic",
                "rep_id": rep_id,
                "x0_rep": x0_rep,
                "r": float(r),
                "expected_period": int(expected_period),
                "burn_in_start_index": int(burn_in_start_index),
                "phase_offset": int(used_phase_offset),
                "used_burn_in": bool(used_burn_in),
            },
        }

    # Start chaotic trajectories from the sampled x0
    for r in ensemble_chaotic_rs:
        vals = logistic_sequence(r, x0_rep, TOTAL_SEQUENCE_STATES)
        data_name = f"Logistic r={r}"
        rep_inputs[data_name] = {
            "sequence": quantize_logistic_window(vals),
            "metadata": {
                "input_class": "chaotic",
                "rep_id": rep_id,
                "x0_rep": x0_rep,
                "r": float(r),
                "burn_in_start_index": 0,
                "phase_offset": 0,
                "used_burn_in": False,
            },
        }

    # Perturb Lorenz x0 and quantize the x-coordinate
    sol_rep = solve_ivp(
        lorenz,
        t_span,
        [lorenz_x0_rep, lorenz_y0_rep, lorenz_z0_rep],
        args=(sigma, rho, beta),
        t_eval=t_eval,
    )
    if not sol_rep.success:
        raise RuntimeError(f"Lorenz solve failed for rep_id={rep_id}: {sol_rep.message}")
    if sol_rep.y.shape[-1] != TOTAL_SEQUENCE_STATES:
        raise ValueError(
            f"Expected {TOTAL_SEQUENCE_STATES} Lorenz samples for rep_id={rep_id}, got {sol_rep.y.shape[-1]}."
        )
    rep_inputs["Lorenz"] = {
        "sequence": quantize_lorenz_x_values(sol_rep.y[0]),
        "metadata": {
            "input_class": "lorenz",
            "rep_id": rep_id,
            "x0_rep": x0_rep,
            "lorenz_x0_rep": lorenz_x0_rep,
            "lorenz_y0_rep": lorenz_y0_rep,
            "lorenz_z0_rep": lorenz_z0_rep,
        },
    }

    # Validate token sequences before model inference
    for data_name in input_data_order_full:
        sequence = np.asarray(rep_inputs[str(data_name)]["sequence"], dtype=int)
        if sequence.shape != (TOTAL_SEQUENCE_STATES,):
            raise ValueError(
                f"{data_name}, rep_id={rep_id}: expected {TOTAL_SEQUENCE_STATES} states, got {sequence.shape}."
            )
        if not np.all((0 <= sequence) & (sequence <= 999)):
            raise ValueError(f"{data_name}, rep_id={rep_id}: quantized states must be in [0, 999].")

    ensemble_inputs[rep_id] = rep_inputs

if len(ensemble_inputs) != NUM_ENSEMBLE_RUNS:
    raise ValueError(f"Expected {NUM_ENSEMBLE_RUNS} ensemble runs, got {len(ensemble_inputs)}.")
if len(set(constant_tokens.tolist())) <= 1:
    raise ValueError("Expected sampled constant tokens to vary across ensemble replicates.")


Load the same tokenizer and context prefixes. Shuffle token IDs directly, checking
the full token multiset and the unchanged BOS/comma positions before model inference.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)


def _single_token_id(token_str):
    token_ids = tokenizer(token_str, add_special_tokens=False).input_ids
    if len(token_ids) != 1:
        raise ValueError(f"Expected {token_str!r} to tokenize as one token; got {token_ids}.")
    return int(token_ids[0])


COMMA_TOKEN_ID = _single_token_id(",")
# Require one token per state and comma for exact context sizing
NUMERIC_TOKEN_IDS = np.asarray([_single_token_id(f"{i:03d}") for i in range(1000)], dtype=np.int64)
if len(np.unique(NUMERIC_TOKEN_IDS)) != 1000:
    raise ValueError("Expected all 000..999 formatted states to map to distinct single tokens.")


def _first_token_value(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        if x.numel() == 0:
            return None
        return int(x.flatten()[0].item())
    arr = np.asarray(x).reshape(-1)
    if arr.size == 0:
        return None
    return int(arr[0])


def get_token_start_index(
    input_ids=None,
    bos_token_id=None,
    special_tokens_mask=None,
    drop_bos=True,
):
    # Exclude an optional leading BOS from graph and signal nodes
    if not drop_bos:
        return 0

    first_id = _first_token_value(input_ids)
    if first_id is not None and bos_token_id is not None and first_id == int(bos_token_id):
        return 1

    first_mask = _first_token_value(special_tokens_mask)
    if first_mask == 1:
        return 1

    return 0


def _build_input_sequence_from_frac(sequence, frac, total_n=MAX_CONTEXT_STATES):
    # Build comma-delimited prompts at each context fraction
    sequence = np.asarray(sequence, dtype=int)
    if sequence.size < int(total_n):
        raise ValueError(f"Need at least {int(total_n)} states; got {sequence.size}.")
    if np.min(sequence[:int(total_n)]) < 0 or np.max(sequence[:int(total_n)]) > 999:
        raise ValueError("Prompt state tokens must be in [0, 999].")

    frac_use = float(frac)
    if frac_use >= 1.0:
        ctx_len = int(total_n)
    else:
        ctx_len = max(1, int(round(frac_use * int(total_n))))
    ctx_len = min(int(total_n), ctx_len)
    context_window = sequence[:ctx_len]
    input_sequence = ",".join([f"{x:03d}" for x in context_window]) + ","
    return input_sequence, int(ctx_len)


def _post_bos_token_ids(input_ids, token_start):
    flat_ids = input_ids.reshape(-1)
    return flat_ids[int(token_start):]


def tokenize_and_validate_diagnostic_prompt(prompt, expected_post_bos_tokens=None):
    if not prompt.endswith(","):
        raise ValueError("Diagnostic prompt must end with a comma.")

    encoded = tokenizer(
        prompt,
        add_special_tokens=TOKENIZATION_ADD_SPECIAL_TOKENS,
        return_special_tokens_mask=True,
        return_tensors="pt",
    )
    input_ids = encoded.input_ids[0]
    special_tokens_mask = encoded.special_tokens_mask[0]
    token_start = get_token_start_index(
        input_ids=input_ids,
        bos_token_id=tokenizer.bos_token_id,
        special_tokens_mask=special_tokens_mask,
        drop_bos=DROP_BOS,
    )
    post_bos_ids = _post_bos_token_ids(input_ids, token_start)
    seq_len = int(post_bos_ids.shape[0])
    if expected_post_bos_tokens is not None and seq_len != int(expected_post_bos_tokens):
        raise ValueError(
            f"Expected {int(expected_post_bos_tokens)} post-BOS content tokens, got {seq_len}."
        )
    if seq_len < 2:
        raise ValueError("Need at least two graph tokens.")
    if int(post_bos_ids[-1].item()) != COMMA_TOKEN_ID:
        decoded_final = tokenizer.decode([int(post_bos_ids[-1].item())], clean_up_tokenization_spaces=False)
        raise ValueError(f"Final post-BOS token is not comma; got {decoded_final!r}.")
    return {
        "encoded": encoded,
        "token_start": int(token_start),
        "seq_len": int(seq_len),
        "final_token_is_comma": True,
    }


reference_sequence = np.arange(MAX_CONTEXT_STATES, dtype=int) % 1000
context_state_counts = np.zeros(len(context_fracs), dtype=np.int64)
context_token_counts = np.zeros(len(context_fracs), dtype=np.int64)
# Measure exact post-BOS lengths with a reference prompt
for context_idx, frac in enumerate(context_fracs):
    reference_prompt, ctx_len = _build_input_sequence_from_frac(
        reference_sequence,
        frac,
        total_n=MAX_CONTEXT_STATES,
    )
    prompt_info = tokenize_and_validate_diagnostic_prompt(reference_prompt)
    context_state_counts[context_idx] = int(ctx_len)
    context_token_counts[context_idx] = int(prompt_info["seq_len"])

expected_context_state_counts = np.maximum(
    1,
    np.rint(context_fracs * MAX_CONTEXT_STATES).astype(np.int64),
)
if not np.array_equal(context_state_counts, expected_context_state_counts):
    raise ValueError(
        f"Expected exact context state counts {expected_context_state_counts.tolist()}, "
        f"got {context_state_counts.tolist()}."
    )
if not np.array_equal(context_token_counts, 2 * context_state_counts):
    raise ValueError("Expected one numeric token and one comma token per context state.")

k_cutoff_by_context = context_token_counts // 2

if not np.all(np.diff(context_token_counts) > 0):
    raise ValueError(f"context_token_counts must be strictly increasing; got {context_token_counts.tolist()}.")
if int(context_token_counts[-1]) != 2000:
    raise ValueError(f"Full context must be 2000 post-BOS content tokens; got {context_token_counts[-1]}.")


def _token_digest(token_ids):
    values = token_ids.detach().cpu().numpy().astype("<i8", copy=False)
    return hashlib.sha256(values.tobytes()).hexdigest()


def build_shuffled_prompt_info(sequence, frac, data_name, rep_id, expected_post_bos_tokens):
    # Tokenize the original prefix once, then permute only its numerical token IDs
    prompt, ctx_len = _build_input_sequence_from_frac(sequence, frac, total_n=MAX_CONTEXT_STATES)
    info = tokenize_and_validate_diagnostic_prompt(prompt, expected_post_bos_tokens)
    encoded = info["encoded"]
    original_ids = encoded["input_ids"][0].clone()
    token_start = info["token_start"]
    original_content = original_ids[token_start:]
    expected_numeric_ids = torch.as_tensor(NUMERIC_TOKEN_IDS[np.asarray(sequence[:ctx_len], dtype=int)])
    if original_content.numel() != 2 * ctx_len:
        raise ValueError("Expected exactly one numeric token and one comma per state.")
    if not torch.equal(original_content[::2], expected_numeric_ids):
        raise ValueError("Original prefix does not tokenize as the expected numerical state tokens.")
    if not torch.all(original_content[1::2] == COMMA_TOKEN_ID):
        raise ValueError("Original commas are not at the expected alternating positions.")

    # Full-family index and state count make permutations independent of task/resume order
    family_index = list(input_data_order_full.astype(str)).index(str(data_name))
    seed_sequence = np.random.SeedSequence([SHUFFLE_SEED, family_index, int(rep_id), int(ctx_len)])
    rng = np.random.Generator(np.random.PCG64(seed_sequence))
    permutation = torch.as_tensor(rng.permutation(ctx_len), dtype=torch.long)
    shuffled_ids = original_ids.clone()
    shuffled_ids[token_start::2] = original_content[::2][permutation]
    shuffled_content = shuffled_ids[token_start:]

    if not torch.equal(torch.sort(original_ids).values, torch.sort(shuffled_ids).values):
        raise ValueError("Shuffle changed the exact model-token multiset.")
    if not torch.equal(shuffled_ids[:token_start], original_ids[:token_start]):
        raise ValueError("Shuffle changed the leading special token.")
    if not torch.equal(shuffled_content[1::2], original_content[1::2]):
        raise ValueError("Shuffle moved a comma.")
    # Do not reject identity permutations: constants must remain unchanged
    encoded["input_ids"] = shuffled_ids.unsqueeze(0)
    info["audit"] = {
        "unique_numeric_token_count": int(torch.unique(original_content[::2]).numel()),
        "unique_content_token_count": int(torch.unique(original_content).numel()),
        "unique_model_token_count": int(torch.unique(original_ids).numel()),
        "fraction_numeric_positions_changed": float(
            (shuffled_content[::2] != original_content[::2]).float().mean().item()
        ),
        "original_token_sha256": _token_digest(original_ids),
        "shuffled_token_sha256": _token_digest(shuffled_ids),
        "token_multiset_verified": True,
    }
    return info


# Record the actual generated trajectories and numerical-token mapping for reproducibility
sequence_hasher = hashlib.sha256()
for name in input_names:
    for rep_id in run_ids:
        sequence_hasher.update(np.asarray(
            ensemble_inputs[int(rep_id)][str(name)]["sequence"], dtype="<i8"
        ).tobytes())
ensemble_sequence_sha256 = sequence_hasher.hexdigest()
numeric_token_ids_sha256 = hashlib.sha256(NUMERIC_TOKEN_IDS.astype("<i8").tobytes()).hexdigest()
generation_notebook_path = NOTEBOOK_DIR / "Generate-shuffled-layer-averaged-diagnostics-across-context-lengths.ipynb"
generation_notebook = json.loads(generation_notebook_path.read_text())
generation_source_sha256 = hashlib.sha256("\n\n".join(
    "".join(cell["source"]) for cell in generation_notebook["cells"] if cell["cell_type"] == "code"
).encode()).hexdigest()


Compute the graph-spectral diagnostics for one layer from attention and hidden-state tensors.

In [ ]:
def compute_all_layer_diagnostics_from_attention_and_hidden(
    attention_layer,
    hidden_state_layer,
    input_ids,
    layer_idx,
    expected_seq_len,
    k_cutoff=None,
    bos_token_id=None,
    special_tokens_mask=None,
    drop_bos=True,
    renormalize_after_drop=True,
    head_weighting="mass",
):
    # Remove the same leading BOS from attention and hidden states
    token_start = get_token_start_index(
        input_ids=input_ids,
        bos_token_id=bos_token_id,
        special_tokens_mask=special_tokens_mask,
        drop_bos=drop_bos,
    )

    A = attention_layer[0].float()
    A = A[:, token_start:, token_start:]
    seq_len = int(A.shape[-1])
    if seq_len != int(expected_seq_len):
        raise ValueError(f"Layer {layer_idx}: expected seq_len={expected_seq_len}, got {seq_len}.")
    if seq_len < 2:
        raise ValueError("Need at least two graph tokens.")

    if renormalize_after_drop and token_start > 0:
        row_sums = A.sum(dim=-1, keepdim=True)
        row_sums = torch.where(row_sums == 0, torch.ones_like(row_sums), row_sums)
        A = A / row_sums

    # Symmetrize heads and combine them into one weighted graph
    if k_cutoff is None:
        k_cutoff = seq_len // 2
    k_cutoff = int(k_cutoff)
    if k_cutoff < 0 or k_cutoff > seq_len:
        raise ValueError(f"Layer {layer_idx}: invalid k_cutoff={k_cutoff} for seq_len={seq_len}.")

    W_heads = 0.5 * (A + A.transpose(-1, -2))
    if head_weighting == "uniform":
        alpha = torch.full((A.shape[0],), 1.0 / A.shape[0], device=A.device, dtype=A.dtype)
    elif head_weighting == "mass":
        s = A.sum(dim=(1, 2))
        s_sum = s.sum()
        if s_sum <= 0:
            alpha = torch.full((A.shape[0],), 1.0 / A.shape[0], device=A.device, dtype=A.dtype)
        else:
            alpha = s / s_sum
    else:
        raise ValueError("head_weighting must be 'mass' or 'uniform'.")

    Wbar = (alpha[:, None, None] * W_heads).sum(dim=0)
    deg = Wbar.sum(dim=1)

    L = torch.diag(deg) - Wbar
    _, U = torch.linalg.eigh(L)

    # Project hidden states onto graph Fourier modes
    X = hidden_state_layer[0].float()
    X = X[token_start:, :].to(U.device)
    if X.shape[0] != seq_len:
        raise ValueError(
            f"Layer {layer_idx}: hidden-state length mismatch after BOS drop; X={X.shape[0]}, graph={seq_len}."
        )

    X_hat = U.T @ X
    energy = (X_hat ** 2).sum(dim=1)
    total_energy = energy.sum()
    if total_energy <= 0 or not torch.isfinite(total_energy):
        raise ValueError(f"Layer {layer_idx}: nonpositive or nonfinite total signal energy: {total_energy}.")
    p = energy / total_energy
    positive_p = p[p > 0]
    spectral_entropy = -torch.sum(positive_p * torch.log(positive_p))
    effective_active_modes = torch.exp(spectral_entropy)

    # Original top-50% HFER; retain the same graph Fourier basis and cutoff
    hfer = float(p[k_cutoff:].sum().detach().cpu().item())

    # The normalized eigensolve is still needed for the Fiedler value
    inv_sqrt_deg = torch.where(deg > 0, deg.pow(-0.5), torch.zeros_like(deg))
    Wbar_norm = inv_sqrt_deg[:, None] * Wbar * inv_sqrt_deg[None, :]
    L_sym = torch.eye(seq_len, device=Wbar.device, dtype=Wbar.dtype) - Wbar_norm
    eigvals_sym = torch.linalg.eigvalsh(L_sym)
    result = {
        "effective_active_modes": float(effective_active_modes.detach().cpu().item()),
        "normalized_fiedler_value": float(eigvals_sym[1].detach().cpu().item()),
        "hfer": hfer,
        "seq_len": int(seq_len),
        "k_cutoff": int(k_cutoff),
        "token_start": int(token_start),
        "attention_layer_index": int(layer_idx),
        "signal_layer_index": int(layer_idx),
    }

    del A, W_heads, Wbar, deg, L, U, X, X_hat, energy, p, inv_sqrt_deg, Wbar_norm, L_sym, eigvals_sym
    return result


Create, validate, load, and save checkpoint arrays for resumable sweeps.

In [ ]:
METRIC_KEYS = (
    "active_modes_layeravg_per_run",
    "fiedler_layeravg_per_run",
    "hfer_layeravg_per_run",
)
AUDIT_INT_KEYS = (
    "unique_numeric_token_count", "unique_content_token_count", "unique_model_token_count",
)
AUDIT_HASH_KEYS = ("original_token_sha256", "shuffled_token_sha256")


def _expected_result_shape():
    return (len(input_names), len(run_ids), len(context_fracs))


def _initialize_result_arrays():
    shape = _expected_result_shape()
    arrays = {key: np.full(shape, np.nan, dtype=np.float64) for key in METRIC_KEYS}
    arrays.update({key: np.full(shape, -1, dtype=np.int64) for key in AUDIT_INT_KEYS})
    arrays.update({key: np.full(shape, "", dtype="U64") for key in AUDIT_HASH_KEYS})
    arrays["fraction_numeric_positions_changed"] = np.full(shape, np.nan, dtype=np.float64)
    arrays["token_multiset_verified"] = np.zeros(shape, dtype=bool)
    arrays["completed_mask"] = np.zeros(shape, dtype=bool)
    return arrays


def _experiment_metadata(num_layers):
    return {
        "npz_schema_version": np.asarray(NPZ_SCHEMA_VERSION),
        "model_name": np.asarray(MODEL_NAME),
        "num_layers": np.asarray(num_layers),
        "num_ensemble_runs": np.asarray(len(run_ids)),
        "input_names": input_names,
        "input_labels": input_labels,
        "input_class_labels": input_class_labels,
        "run_ids": run_ids,
        "context_fracs": context_fracs,
        "context_state_counts": context_state_counts,
        "context_token_counts": context_token_counts,
        "k_cutoff_by_context": k_cutoff_by_context,
        "seed": np.asarray(SEED),
        "ensemble_seed": np.asarray(ENSEMBLE_SEED),
        "shuffle_seed": np.asarray(SHUFFLE_SEED),
        "shuffle_method": np.asarray(SHUFFLE_METHOD),
        "shuffles_per_realization_context": np.asarray(1),
        "shuffle_rng": np.asarray("PCG64/SeedSequence[seed,family_index,rep_id,context_states]"),
        "max_context_states": np.asarray(MAX_CONTEXT_STATES),
        "total_sequence_states": np.asarray(TOTAL_SEQUENCE_STATES),
        "drop_bos": np.asarray(DROP_BOS),
        "renormalize_after_drop": np.asarray(RENORMALIZE_AFTER_DROP),
        "head_weighting": np.asarray(HEAD_WEIGHTING),
        "tokenization_add_special_tokens": np.asarray(TOKENIZATION_ADD_SPECIAL_TOKENS),
        "hfer_tail_fraction": np.asarray(0.5),
        "signal_layer_indexing": np.asarray("outputs.hidden_states[layer_idx]"),
        "ensemble_sequence_sha256": np.asarray(ensemble_sequence_sha256),
        "numeric_token_ids_sha256": np.asarray(numeric_token_ids_sha256),
        "generation_source_sha256": np.asarray(generation_source_sha256),
        "x0_samples": x0_samples,
        "constant_tokens": constant_tokens,
    }


def _validate_result_arrays(arrays, require_complete=False):
    for key, template in _initialize_result_arrays().items():
        if key not in arrays or arrays[key].shape != template.shape or arrays[key].dtype != template.dtype:
            raise ValueError(f"Invalid shape or dtype for {key}.")
    completed = arrays["completed_mask"]
    if require_complete and not completed.all():
        raise ValueError(f"Sweep incomplete: {completed.sum()} / {completed.size} tasks.")
    for key in METRIC_KEYS + ("fraction_numeric_positions_changed",):
        if not np.isfinite(arrays[key][completed]).all():
            raise ValueError(f"Completed tasks contain nonfinite {key}.")
    if not arrays["token_multiset_verified"][completed].all():
        raise ValueError("Completed tasks must pass exact-token multiset verification.")
    for key in AUDIT_INT_KEYS:
        if not (arrays[key][completed] > 0).all():
            raise ValueError(f"Completed tasks contain invalid {key}.")
    for key in AUDIT_HASH_KEYS:
        if not (np.char.str_len(arrays[key][completed]) == 64).all():
            raise ValueError(f"Completed tasks contain invalid {key}.")
    bounds = {
        "active_modes_layeravg_per_run": (1.0 - 1e-5, np.broadcast_to(context_token_counts, completed.shape)[completed] + 1e-3),
        "fiedler_layeravg_per_run": (-1e-4, 2.0 + 1e-4),
        "hfer_layeravg_per_run": (-1e-6, 1.0 + 1e-6),
        "fraction_numeric_positions_changed": (0.0, 1.0),
    }
    for key, (lower, upper) in bounds.items():
        values = arrays[key][completed]
        if not ((values >= lower) & (values <= upper)).all():
            raise ValueError(f"Completed tasks contain out-of-range {key}.")
    for input_idx, name in enumerate(input_names):
        if str(name) == "Constant sampled":
            mask = completed[input_idx]
            if not (arrays["fraction_numeric_positions_changed"][input_idx][mask] == 0).all():
                raise ValueError("Constant trajectories must remain unchanged.")
            if not np.array_equal(arrays["original_token_sha256"][input_idx][mask],
                                  arrays["shuffled_token_sha256"][input_idx][mask]):
                raise ValueError("Constant token hashes must match.")


def _validate_checkpoint_compatibility(data, num_layers):
    expected_metadata = _experiment_metadata(num_layers)
    expected_metadata["checkpoint_schema_version"] = np.asarray(CHECKPOINT_SCHEMA_VERSION)
    for key, expected in expected_metadata.items():
        if key not in data or not np.array_equal(data[key], expected):
            raise ValueError(f"Checkpoint {key} mismatch; use a separate cache for changed settings.")


def load_or_initialize_checkpoint(num_layers):
    if RESUME_FROM_CHECKPOINT and CHECKPOINT_PATH.exists():
        with np.load(CHECKPOINT_PATH, allow_pickle=False) as data:
            _validate_checkpoint_compatibility(data, num_layers)
            arrays = {key: data[key].copy() for key in _initialize_result_arrays()}
        _validate_result_arrays(arrays)
        print(f"Resuming {arrays['completed_mask'].sum()} completed tasks from {CHECKPOINT_PATH}")
        return arrays
    return _initialize_result_arrays()


def _save_npz_atomic(path, payload):
    # Preserve the previous checkpoint if serialization is interrupted
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as handle:
        np.savez_compressed(handle, **payload)
    temporary.replace(path)


def save_checkpoint(arrays, num_layers):
    _validate_result_arrays(arrays)
    _save_npz_atomic(CHECKPOINT_PATH, {
        **_experiment_metadata(num_layers), **arrays,
        "checkpoint_schema_version": np.asarray(CHECKPOINT_SCHEMA_VERSION),
    })


Load the model with eager attention and capture layer/device metadata.

In [ ]:
# Use eager attention to return full attention tensors
model_attn = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager",
)
model_attn.eval()

num_layers = int(model_attn.config.num_hidden_layers)
if num_layers != EXPECTED_NUM_LAYERS:
    raise ValueError(f"Expected model to have {EXPECTED_NUM_LAYERS} layers, got {num_layers}.")

layer_indices = np.arange(num_layers, dtype=np.int64)
device = next(model_attn.parameters()).device


Run or resume the full sweep, averaging diagnostics across layers and checkpointing progress.

In [ ]:
arrays = load_or_initialize_checkpoint(num_layers=num_layers)
completed_since_checkpoint = 0

# Task grid: input family x realization x context length
tasks = [
    (input_axis_idx, str(data_name), run_axis_idx, int(rep_id), context_idx)
    for input_axis_idx, data_name in enumerate(input_names)
    for run_axis_idx, rep_id in enumerate(run_ids)
    for context_idx in range(len(context_fracs))
]

checkpoint_every = max(1, int(CHECKPOINT_EVERY_N_TASKS))
initial_completed_total = int(arrays["completed_mask"].sum())
next_checkpoint_completed = (
    (initial_completed_total // checkpoint_every) + 1
) * checkpoint_every

for input_axis_idx, data_name, run_axis_idx, rep_id, context_idx in tqdm(
    tasks,
    desc=f"Shuffled three diagnostics ({NUM_ENSEMBLE_RUNS} realizations) vs context",
    miniters=checkpoint_every,
):
    # Skip tasks saved in a compatible checkpoint
    if arrays["completed_mask"][input_axis_idx, run_axis_idx, context_idx]:
        continue

    sequence = np.asarray(ensemble_inputs[int(rep_id)][data_name]["sequence"], dtype=int)
    prompt_info = build_shuffled_prompt_info(
        sequence, context_fracs[context_idx], data_name, rep_id,
        expected_post_bos_tokens=int(context_token_counts[context_idx]),
    )
    encoded = prompt_info["encoded"]
    model_inputs = {
        key: value.to(device)
        for key, value in encoded.items()
        if key in {"input_ids", "attention_mask", "position_ids", "token_type_ids"}
    }
    input_ids_for_diagnostics = model_inputs["input_ids"][0]
    special_tokens_mask = (
        encoded["special_tokens_mask"][0].to(device)
        if "special_tokens_mask" in encoded
        else None
    )
    expected_seq_len = int(context_token_counts[context_idx])
    expected_k_cutoff = int(k_cutoff_by_context[context_idx])

    # Request attention and hidden states for each layer
    with torch.no_grad():
        outputs = model_attn(
            **model_inputs,
            output_attentions=True,
            output_hidden_states=True,
            return_dict=True,
        )

    if outputs.attentions is None or outputs.hidden_states is None:
        raise ValueError("Model outputs must include attentions and hidden_states.")
    if len(outputs.attentions) != num_layers:
        raise ValueError(f"Expected {num_layers} attention tensors, got {len(outputs.attentions)}.")
    if len(outputs.hidden_states) != num_layers + 1:
        raise ValueError(f"Expected {num_layers + 1} hidden-state tensors, got {len(outputs.hidden_states)}.")

    active_modes_sum = 0.0
    fiedler_sum = 0.0
    hfer_sum = 0.0

    # Accumulate and average each diagnostic across layers
    for layer_idx in range(num_layers):
        bundle = compute_all_layer_diagnostics_from_attention_and_hidden(
            outputs.attentions[layer_idx],
            outputs.hidden_states[layer_idx],
            input_ids=input_ids_for_diagnostics,
            layer_idx=layer_idx,
            expected_seq_len=expected_seq_len,
            k_cutoff=expected_k_cutoff,
            bos_token_id=tokenizer.bos_token_id,
            special_tokens_mask=special_tokens_mask,
            drop_bos=DROP_BOS,
            renormalize_after_drop=RENORMALIZE_AFTER_DROP,
            head_weighting=HEAD_WEIGHTING,
        )
        active_modes_sum += float(bundle["effective_active_modes"])
        fiedler_sum += float(bundle["normalized_fiedler_value"])
        hfer_sum += float(bundle["hfer"])
        del bundle

    active_modes_layeravg = active_modes_sum / num_layers
    fiedler_layeravg = fiedler_sum / num_layers
    hfer_layeravg = hfer_sum / num_layers

    arrays["active_modes_layeravg_per_run"][input_axis_idx, run_axis_idx, context_idx] = active_modes_layeravg
    arrays["fiedler_layeravg_per_run"][input_axis_idx, run_axis_idx, context_idx] = fiedler_layeravg
    arrays["hfer_layeravg_per_run"][input_axis_idx, run_axis_idx, context_idx] = hfer_layeravg
    for key, value in prompt_info["audit"].items():
        arrays[key][input_axis_idx, run_axis_idx, context_idx] = value
    arrays["completed_mask"][input_axis_idx, run_axis_idx, context_idx] = True
    completed_since_checkpoint += 1

    # Release large tensors between tasks
    del outputs, model_inputs, encoded, prompt_info, input_ids_for_diagnostics, special_tokens_mask
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    completed_total = int(arrays["completed_mask"].sum())
    if completed_total >= next_checkpoint_completed:
        save_checkpoint(arrays, num_layers=num_layers)
        completed_since_checkpoint = 0
        next_checkpoint_completed = (
            (completed_total // checkpoint_every) + 1
        ) * checkpoint_every

if completed_since_checkpoint > 0 or not CHECKPOINT_PATH.exists():
    save_checkpoint(arrays, num_layers=num_layers)


Validate completed arrays, summarize over runs, and save the visualization cache.

In [ ]:
def _summarize_over_runs(per_run):
    mean = np.mean(per_run, axis=1)
    std = np.std(per_run, axis=1, ddof=1) if per_run.shape[1] > 1 else np.zeros_like(mean)
    sem = std / np.sqrt(per_run.shape[1])
    return mean, std, sem


_validate_result_arrays(arrays, require_complete=True)
summaries = {}
for metric, key in zip(("active_modes", "fiedler", "hfer"), METRIC_KEYS):
    mean, std, sem = _summarize_over_runs(arrays[key])
    summaries.update({f"{metric}_mean": mean, f"{metric}_std": std, f"{metric}_sem": sem})

_save_npz_atomic(OUTPUT_PATH, {**_experiment_metadata(num_layers), **arrays, **summaries})
print(f"Saved {OUTPUT_PATH}")
print(f"Verified exact token preservation for all {arrays['completed_mask'].size} tasks.")
print("Arrays retain per-realization diagnostics, unique-token counts, and original/shuffled token hashes.")
